# Dataset Preparation: Rename File Gambar ALPR Lampung

**Project:** Automatic License Plate Recognition (ALPR) Kabupaten/Kota Lampung  
**Tujuan Notebook:** Merapikan dan menyeragamkan nama file gambar dataset agar siap digunakan untuk proses anotasi di Roboflow dan pembuatan ground truth spreadsheet.

---

## Latar Belakang

Dataset ini merupakan kumpulan gambar kendaraan yang digunakan untuk melatih model deteksi plat nomor kendaraan berbasis kode wilayah Lampung (kode plat: BE). Gambar dikumpulkan dari berbagai sumber dan sesi pengambilan data yang sesinya sudah tidak dapat diidentifikasi secara pasti.

## Mengapa Nama File Tidak Mengandung Informasi Plat atau Wilayah?

Praktik terbaik dalam manajemen dataset machine learning mensyaratkan bahwa nama file bersifat **netral** dan tidak mengandung label atau metadata substantif. Alasannya adalah:

1. **Menghindari data leakage**: Jika nama file mengandung nomor plat atau kode wilayah, model berpotensi mendapat informasi label secara implisit sebelum proses inferensi.
2. **Konsistensi pipeline**: Sistem anotasi seperti Roboflow bekerja optimal dengan nama file yang seragam dan tidak ambigu.
3. **Fleksibilitas relabeling**: Ground truth yang tersimpan di file nama sulit diubah tanpa merename ulang seluruh dataset. Dengan menyimpan label di spreadsheet terpisah, relabeling menjadi jauh lebih mudah.
4. **Privasi**: Nomor plat kendaraan termasuk data pribadi. Menyembunyikannya dari nama file mengurangi risiko eksposur yang tidak disengaja.

## Ground Truth Disimpan Terpisah

Seluruh metadata substantif seperti nomor plat asli, kode wilayah, nama kabupaten/kota, kualitas gambar, dan catatan tambahan akan disimpan di **spreadsheet terpisah** (misalnya Google Sheets atau CSV ground truth), bukan di nama file. Notebook ini hanya menghasilkan **mapping CSV** yang mencatat nama file lama dan nama file baru.

## Format Nama File Baru

```
alpr_lampung_000001.jpg
alpr_lampung_000002.jpg
alpr_lampung_000003.jpg
```

- `alpr_lampung` : identifier project
- `000001`       : nomor urut global 6 digit (zero-padded)
- `.jpg`         : ekstensi diseragamkan

Karena sesi pengambilan data tidak dapat diidentifikasi, **kode sesi tidak digunakan**. Penomoran bersifat global dan urut berdasarkan nama file asli.

---

## Sel 1: Instalasi Library yang Diperlukan

Library standar Python seperti `os`, `shutil`, `pathlib`, dan `csv` sudah tersedia di Google Colab tanpa instalasi tambahan. Satu-satunya library eksternal yang mungkin diperlukan adalah `pillow-heif` untuk membaca file berformat HEIC (format foto dari perangkat Apple). Library `Pillow` sendiri sudah tersedia secara default di Colab.

Instalasi dilakukan hanya jika belum tersedia di lingkungan Colab saat ini.

In [ ]:
# Instalasi pillow-heif untuk mendukung konversi file HEIC ke JPG
# Library ini dibutuhkan jika dataset mengandung foto dari perangkat iPhone/iPad
try:
    import pillow_heif
    print("pillow-heif sudah tersedia.")
except ImportError:
    print("Menginstal pillow-heif...")
    import subprocess
    subprocess.run(["pip", "install", "pillow-heif", "-q"], check=True)
    print("Instalasi pillow-heif selesai.")

# Verifikasi Pillow
from PIL import Image
print(f"Pillow tersedia: versi {Image.__version__ if hasattr(Image, '__version__') else 'OK'}")

pillow-heif sudah tersedia.
Pillow tersedia: versi 11.3.0


## Sel 2: Import Library

Seluruh library yang dibutuhkan dalam proses ini diimpor pada sel berikut. Penjelasan masing-masing:

- `os` dan `pathlib.Path`: untuk navigasi dan manipulasi path sistem file
- `shutil`: untuk menyalin file dari lokasi asal ke folder output
- `csv`: untuk membuat file mapping CSV
- `PIL.Image` dan `pillow_heif`: untuk membuka, mengonversi, dan menyimpan gambar
- `datetime`: untuk mencatat timestamp proses
- `google.colab.drive`: untuk mount Google Drive

In [ ]:
import os
import shutil
import csv
from pathlib import Path
from datetime import datetime
from PIL import Image

try:
    import pillow_heif
    pillow_heif.register_heif_opener()  # Daftarkan HEIF/HEIC ke Pillow secara otomatis
    HEIC_SUPPORT = True
    print("Dukungan HEIC/HEIF aktif.")
except ImportError:
    HEIC_SUPPORT = False
    print("Peringatan: pillow-heif tidak tersedia. File HEIC tidak akan bisa dikonversi.")

print("Seluruh library berhasil diimpor.")

Dukungan HEIC/HEIF aktif.
Seluruh library berhasil diimpor.


## Sel 3: Mount Google Drive

Google Drive perlu di-mount terlebih dahulu agar notebook dapat mengakses file yang tersimpan di sana. Setelah sel ini dijalankan, Google Colab akan meminta izin akses ke akun Google Anda. Ikuti instruksi autentikasi yang muncul.

Setelah mount berhasil, Google Drive Anda akan dapat diakses melalui path `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print("Google Drive berhasil di-mount di /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive berhasil di-mount di /content/drive


## Sel 4: Konfigurasi Path dan Parameter

Pada sel ini, seluruh konfigurasi utama notebook didefinisikan dalam satu tempat agar mudah diubah tanpa harus menelusuri seluruh kode.

**Penjelasan parameter:**

- `INPUT_FOLDER_ID`: ID folder Google Drive yang berisi dataset asli. ID ini diambil dari link sharing folder. Contoh: dari link `https://drive.google.com/drive/folders/1cGfR2hJWhe5Z_Z6-6aK4lkWncjbsqR96`, maka ID-nya adalah `1cGfR2hJWhe5Z_Z6-6aK4lkWncjbsqR96`.
- `INPUT_FOLDER_PATH`: Path lengkap folder input yang telah di-mount via Google Drive. Sesuaikan dengan lokasi aktual folder di Drive Anda.
- `OUTPUT_FOLDER_PATH`: Lokasi folder output yang akan dibuat untuk menyimpan file hasil rename.
- `CSV_OUTPUT_PATH`: Path penyimpanan file mapping CSV.
- `PROJECT_PREFIX`: Prefix nama file baru, sesuai format yang disepakati.
- `START_INDEX`: Nomor urut awal penomoran. Ubah jika ingin melanjutkan penomoran dari dataset sebelumnya.
- `DIGIT_LENGTH`: Panjang digit nomor urut (default: 6, menghasilkan `000001`).

> **Catatan Penting:** Sesuaikan `INPUT_FOLDER_PATH` dengan lokasi folder di Google Drive Anda. Folder harus sudah di-add shortcut ke "My Drive" atau berada di dalam "My Drive" Anda.

In [ ]:
# ============================================================
# KONFIGURASI UTAMA - Sesuaikan bagian ini sebelum menjalankan
# ============================================================

# ID folder Google Drive dari link sharing
# Link: https://drive.google.com/drive/folders/1cGfR2hJWhe5Z_Z6-6aK4lkWncjbsqR96
INPUT_FOLDER_ID = "1cGfR2hJWhe5Z_Z6-6aK4lkWncjbsqR96"

# Path folder input di Google Drive (sesuaikan dengan lokasi di Drive Anda)
# Jika folder sudah di-add shortcut ke My Drive, gunakan nama foldernya langsung
# Contoh: "/content/drive/MyDrive/ALPR_Lampung_Dataset"
INPUT_FOLDER_PATH = Path("/content/drive/MyDrive/Dataset-Alpr")

# Folder output untuk menyimpan file hasil rename (tidak boleh sama dengan input)
OUTPUT_FOLDER_PATH = Path("/content/drive/MyDrive/renamed_dataset")

# Path file mapping CSV
CSV_OUTPUT_PATH = Path("/content/drive/MyDrive/renamed_dataset/rename_mapping.csv")

# Format nama file
PROJECT_PREFIX = "alpr_lampung"  # Prefix nama project
START_INDEX    = 1               # Nomor urut awal (ubah jika melanjutkan dari dataset lain)
DIGIT_LENGTH   = 6               # Panjang digit nomor urut (000001 = 6 digit)

# Ekstensi file yang akan diproses (tidak case-sensitive)
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".heic"}

# ============================================================
print("Konfigurasi berhasil dimuat.")
print(f"  Folder Input  : {INPUT_FOLDER_PATH}")
print(f"  Folder Output : {OUTPUT_FOLDER_PATH}")
print(f"  CSV Output    : {CSV_OUTPUT_PATH}")
print(f"  Prefix        : {PROJECT_PREFIX}")
print(f"  Mulai dari    : {START_INDEX}")
print(f"  Format nomor  : {DIGIT_LENGTH} digit")

Konfigurasi berhasil dimuat.
  Folder Input  : /content/drive/MyDrive/Dataset-Alpr
  Folder Output : /content/drive/MyDrive/renamed_dataset
  CSV Output    : /content/drive/MyDrive/renamed_dataset/rename_mapping.csv
  Prefix        : alpr_lampung
  Mulai dari    : 1
  Format nomor  : 6 digit


## Sel 5: Alternatif - Mengakses Folder Shared Drive via ID

Jika folder dataset merupakan **folder shared** (bukan milik akun Anda sendiri), Anda perlu menambahkan shortcut folder tersebut ke "My Drive" terlebih dahulu melalui langkah berikut:

1. Buka link folder: `https://drive.google.com/drive/folders/1cGfR2hJWhe5Z_Z6-6aK4lkWncjbsqR96`
2. Klik kanan folder > **Add shortcut to Drive** > pilih **My Drive**
3. Catat nama folder tersebut, lalu sesuaikan `INPUT_FOLDER_PATH` di Sel 4.

Sel berikut ini mencoba menemukan folder secara otomatis menggunakan ID folder. Jika berhasil, path akan diperbarui otomatis. Jika tidak, Anda harus mengisi `INPUT_FOLDER_PATH` secara manual di Sel 4.

In [ ]:
import subprocess

def find_folder_by_id(folder_id: str, search_base: str = "/content/drive") -> str | None:
    """
    Mencari folder berdasarkan ID Google Drive dengan cara menelusuri
    struktur direktori yang sudah di-mount.
    Mengembalikan path folder jika ditemukan, None jika tidak.
    """
    try:
        result = subprocess.run(
            ["find", search_base, "-name", ".shortcut-targets-by-id", "-type", "d"],
            capture_output=True, text=True, timeout=15
        )
        # Pendekatan langsung: cek apakah folder ID tersedia di path Shared Drives
        candidate_paths = [
            f"/content/drive/Shareddrives",
            f"/content/drive/MyDrive"
        ]
        for base in candidate_paths:
            if Path(base).exists():
                print(f"Ditemukan base path: {base}")
        return None
    except Exception as e:
        print(f"Pencarian otomatis gagal: {e}")
        return None

# Validasi apakah INPUT_FOLDER_PATH yang dikonfigurasi di Sel 4 bisa diakses
print("Memeriksa akses ke folder input...")
if INPUT_FOLDER_PATH.exists() and INPUT_FOLDER_PATH.is_dir():
    print(f"[OK] Folder input ditemukan: {INPUT_FOLDER_PATH}")
else:
    print(f"[PERINGATAN] Folder tidak ditemukan di: {INPUT_FOLDER_PATH}")
    print()
    print("Kemungkinan penyebab:")
    print("  1. Nama folder di konfigurasi tidak sesuai dengan nama di Google Drive.")
    print("  2. Folder belum ditambahkan shortcut ke 'My Drive'.")
    print("  3. Google Drive belum selesai di-mount (coba jalankan ulang Sel 3).")
    print()
    print("Solusi: Buka Google Drive, tambahkan shortcut folder dataset ke My Drive,")
    print(f"lalu perbarui INPUT_FOLDER_PATH di Sel 4 dengan nama folder yang benar.")
    print()
    # Tampilkan isi My Drive sebagai referensi
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        print("Isi My Drive Anda saat ini:")
        for item in sorted(mydrive.iterdir()):
            print(f"  {'[FOLDER]' if item.is_dir() else '[FILE  ]'} {item.name}")

Memeriksa akses ke folder input...
[OK] Folder input ditemukan: /content/drive/MyDrive/Dataset-Alpr


## Sel 6: Membaca Seluruh File Gambar dari Folder Input

Notebook ini akan menelusuri folder input dan mengumpulkan seluruh file dengan ekstensi yang valid (`.jpg`, `.jpeg`, `.png`, `.heic`). Pencarian dilakukan secara **non-rekursif** (hanya satu level folder), namun kode dapat dimodifikasi menjadi rekursif jika dataset memiliki sub-folder.

File diurutkan berdasarkan nama file asli secara alfanumerik agar urutan nomor yang dihasilkan konsisten dan dapat direproduksi di kemudian hari.

In [ ]:
def collect_image_files(folder: Path, extensions: set) -> list[Path]:
    """
    Mengumpulkan seluruh file gambar dalam folder berdasarkan ekstensi yang valid.
    File diurutkan berdasarkan nama file (case-insensitive) agar deterministik.
    """
    files = []
    for f in folder.iterdir():
        if f.is_file() and f.suffix.lower() in extensions:
            files.append(f)
    # Urutkan berdasarkan nama file agar urutan konsisten
    files.sort(key=lambda x: x.name.lower())
    return files


# Jalankan pengumpulan file
image_files = collect_image_files(INPUT_FOLDER_PATH, VALID_EXTENSIONS)

print(f"Total file gambar ditemukan: {len(image_files)}")
print()

# Hitung distribusi ekstensi
from collections import Counter
ext_counter = Counter(f.suffix.lower() for f in image_files)
print("Distribusi ekstensi file:")
for ext, count in sorted(ext_counter.items()):
    print(f"  {ext:10s}: {count} file")

if len(image_files) == 0:
    raise RuntimeError(
        "Tidak ada file gambar yang ditemukan di folder input. "
        "Pastikan INPUT_FOLDER_PATH sudah benar dan folder berisi file gambar."
    )

Total file gambar ditemukan: 185

Distribusi ekstensi file:
  .jpg      : 182 file
  .png      : 3 file


## Sel 7: Menampilkan Contoh Nama File Sebelum Rename

Sebelum proses rename dimulai, beberapa contoh nama file asli ditampilkan sebagai referensi. Hal ini berguna untuk memverifikasi bahwa file yang terbaca sesuai ekspektasi dan folder input sudah benar.

In [ ]:
PREVIEW_COUNT = 10  # Jumlah file yang ditampilkan sebagai preview

print(f"Preview {min(PREVIEW_COUNT, len(image_files))} file pertama (dari total {len(image_files)}):")
print("-" * 60)
for i, f in enumerate(image_files[:PREVIEW_COUNT]):
    print(f"  [{i+1:>4}] {f.name}")

if len(image_files) > PREVIEW_COUNT:
    print(f"  ... dan {len(image_files) - PREVIEW_COUNT} file lainnya.")
print("-" * 60)

# Perkirakan nama file output pertama dan terakhir
first_new = f"{PROJECT_PREFIX}_{str(START_INDEX).zfill(DIGIT_LENGTH)}.jpg"
last_new  = f"{PROJECT_PREFIX}_{str(START_INDEX + len(image_files) - 1).zfill(DIGIT_LENGTH)}.jpg"
print()
print(f"Perkiraan nama file baru:")
print(f"  Pertama : {first_new}")
print(f"  Terakhir: {last_new}")

Preview 10 file pertama (dari total 185):
------------------------------------------------------------
  [   1] IMG_20260523_175827.jpg
  [   2] IMG_20260523_175855.jpg
  [   3] IMG_20260523_175917.jpg
  [   4] IMG_20260523_175943.jpg
  [   5] IMG_20260523_180437.jpg
  [   6] IMG_20260523_180513.jpg
  [   7] IMG_20260523_181337.png
  [   8] IMG_20260523_181410.png
  [   9] IMG_20260523_181508.jpg
  [  10] IMG_20260523_181555.jpg
  ... dan 175 file lainnya.
------------------------------------------------------------

Perkiraan nama file baru:
  Pertama : alpr_lampung_000001.jpg
  Terakhir: alpr_lampung_000185.jpg


## Sel 8: Membuat Folder Output

Folder output dibuat jika belum ada. Notebook ini menggunakan prinsip **non-destruktif**: file asli di folder input tidak akan diubah atau dihapus. Seluruh proses dilakukan dengan menyalin file ke folder output baru.

Jika folder output sudah ada (misalnya dari eksekusi sebelumnya), notebook akan melaporkannya dan melanjutkan proses. Pengguna dapat memilih apakah akan menghapus isi folder output terlebih dahulu atau tidak.

In [ ]:
# Cek apakah folder output sudah ada
if OUTPUT_FOLDER_PATH.exists():
    existing_files = list(OUTPUT_FOLDER_PATH.iterdir())
    print(f"[INFO] Folder output sudah ada: {OUTPUT_FOLDER_PATH}")
    print(f"       Berisi {len(existing_files)} item.")
    print()
    print("Pilihan:")
    print("  - Lanjutkan tanpa menghapus isi yang sudah ada (file baru akan ditambahkan).")
    print("  - Jika ingin membersihkan folder output, ubah CLEAR_OUTPUT_FOLDER = True di bawah.")
    print()
else:
    print(f"[INFO] Folder output belum ada, akan dibuat baru.")

# Ganti True jika ingin menghapus isi folder output sebelum proses dimulai
CLEAR_OUTPUT_FOLDER = False

if CLEAR_OUTPUT_FOLDER and OUTPUT_FOLDER_PATH.exists():
    shutil.rmtree(OUTPUT_FOLDER_PATH)
    print("[INFO] Folder output berhasil dikosongkan.")

OUTPUT_FOLDER_PATH.mkdir(parents=True, exist_ok=True)
print(f"[OK] Folder output siap: {OUTPUT_FOLDER_PATH}")

[INFO] Folder output belum ada, akan dibuat baru.
[OK] Folder output siap: /content/drive/MyDrive/renamed_dataset


## Sel 9: Fungsi Pembantu untuk Proses Copy dan Konversi

Sel ini mendefinisikan fungsi-fungsi utama yang digunakan dalam proses rename:

1. **`convert_and_save_as_jpg`**: Membuka gambar menggunakan Pillow dan menyimpannya sebagai file JPG. Fungsi ini menangani konversi mode warna (misalnya RGBA ke RGB) yang diperlukan sebelum menyimpan sebagai JPG.
2. **`copy_as_jpg`**: Fungsi utama yang memutuskan apakah sebuah file perlu dikonversi (HEIC) atau cukup disalin dan dikonversi ke JPG secara langsung.
3. **`generate_new_filename`**: Menghasilkan nama file baru berdasarkan prefix, nomor urut, dan panjang digit yang dikonfigurasi.

In [ ]:
def generate_new_filename(prefix: str, index: int, digit_length: int) -> str:
    """
    Menghasilkan nama file baru dengan format:
    {prefix}_{nomor_urut_zero_padded}.jpg
    Contoh: alpr_lampung_000001.jpg
    """
    return f"{prefix}_{str(index).zfill(digit_length)}.jpg"


def convert_and_save_as_jpg(src_path: Path, dst_path: Path, quality: int = 95) -> None:
    """
    Membuka gambar dari src_path menggunakan Pillow dan menyimpannya
    sebagai file JPG di dst_path.

    Menangani konversi mode warna:
    - RGBA, LA, P (palette) -> RGB sebelum disimpan sebagai JPG
    - Mode lain langsung disimpan

    Parameter quality=95 menjaga kualitas gambar mendekati lossless
    sambil tetap menghasilkan ukuran file yang wajar.
    """
    with Image.open(src_path) as img:
        # Konversi mode warna jika diperlukan
        if img.mode in ("RGBA", "LA", "P"):
            # Buat background putih untuk menggantikan transparansi
            background = Image.new("RGB", img.size, (255, 255, 255))
            if img.mode == "P":
                img = img.convert("RGBA")
            if img.mode in ("RGBA", "LA"):
                background.paste(img, mask=img.split()[-1])
            img = background
        elif img.mode != "RGB":
            img = img.convert("RGB")

        img.save(dst_path, format="JPEG", quality=quality, optimize=True)


def process_single_file(
    src_path: Path,
    dst_path: Path,
    heic_support: bool
) -> tuple[bool, str]:
    """
    Memproses satu file gambar: menyalinnya ke dst_path sebagai JPG.
    Mengembalikan (success: bool, notes: str).

    Untuk semua format (JPG, PNG, HEIC), file dibuka ulang via Pillow
    untuk memastikan format output konsisten sebagai JPG standar.
    """
    ext = src_path.suffix.lower()

    if ext in (".heic",) and not heic_support:
        return False, "Dilewati: pillow-heif tidak tersedia untuk konversi HEIC."

    try:
        convert_and_save_as_jpg(src_path, dst_path)
        notes = "Berhasil"
        if ext in (".heic",):
            notes = "Berhasil (HEIC -> JPG)"
        elif ext in (".png",):
            notes = "Berhasil (PNG -> JPG)"
        elif ext in (".jpeg",):
            notes = "Berhasil (JPEG -> JPG)"
        return True, notes
    except Exception as e:
        return False, f"Error: {str(e)}"


print("Fungsi pembantu berhasil didefinisikan.")

Fungsi pembantu berhasil didefinisikan.


## Sel 10: Proses Rename - Menyalin dan Merename Seluruh File

Ini adalah sel utama yang menjalankan proses rename secara keseluruhan. Alur prosesnya adalah:

1. Iterasi setiap file gambar yang sudah dikumpulkan (sudah diurutkan).
2. Hasilkan nama file baru berdasarkan nomor urut.
3. Periksa apakah nama file baru sudah ada di folder output (mencegah overwrite).
4. Salin dan konversi file ke folder output.
5. Catat hasil setiap file ke dalam daftar mapping.

Jika terjadi error pada file tertentu, proses **tidak dihentikan**. Error dicatat pada kolom `notes` di mapping CSV dan proses dilanjutkan ke file berikutnya.

Progress ditampilkan setiap 50 file agar eksekusi dapat dipantau.

In [ ]:
mapping_records = []  # Daftar record untuk mapping CSV
success_count  = 0
failure_count  = 0
skipped_count  = 0

print(f"Memulai proses rename untuk {len(image_files)} file...")
print(f"Folder output: {OUTPUT_FOLDER_PATH}")
print("-" * 70)

for idx, src_path in enumerate(image_files):
    current_index = START_INDEX + idx
    new_filename  = generate_new_filename(PROJECT_PREFIX, current_index, DIGIT_LENGTH)
    dst_path      = OUTPUT_FOLDER_PATH / new_filename

    # Validasi: jangan overwrite file yang sudah ada
    if dst_path.exists():
        record = {
            "old_path"          : str(src_path),
            "old_filename"      : src_path.name,
            "new_filename"      : new_filename,
            "original_extension": src_path.suffix.lower(),
            "status"            : "DILEWATI",
            "notes"             : f"File tujuan sudah ada: {dst_path.name}"
        }
        mapping_records.append(record)
        skipped_count += 1
        continue

    # Proses file
    success, notes = process_single_file(src_path, dst_path, HEIC_SUPPORT)

    record = {
        "old_path"          : str(src_path),
        "old_filename"      : src_path.name,
        "new_filename"      : new_filename,
        "original_extension": src_path.suffix.lower(),
        "status"            : "BERHASIL" if success else "GAGAL",
        "notes"             : notes
    }
    mapping_records.append(record)

    if success:
        success_count += 1
    else:
        failure_count += 1
        print(f"  [GAGAL] {src_path.name} -> {new_filename}: {notes}")

    # Tampilkan progress setiap 50 file
    if (idx + 1) % 50 == 0 or (idx + 1) == len(image_files):
        print(f"  Progress: {idx + 1}/{len(image_files)} file diproses "
              f"| Berhasil: {success_count} | Gagal: {failure_count}")

print("-" * 70)
print("Proses rename selesai.")

Memulai proses rename untuk 185 file...
Folder output: /content/drive/MyDrive/renamed_dataset
----------------------------------------------------------------------
  Progress: 50/185 file diproses | Berhasil: 50 | Gagal: 0
  Progress: 100/185 file diproses | Berhasil: 100 | Gagal: 0
  Progress: 150/185 file diproses | Berhasil: 150 | Gagal: 0
  Progress: 185/185 file diproses | Berhasil: 185 | Gagal: 0
----------------------------------------------------------------------
Proses rename selesai.


## Sel 11: Menyimpan Mapping CSV

File mapping CSV berfungsi sebagai **jembatan** antara nama file lama dan nama file baru. File ini sangat penting untuk:

- Melacak asal-usul setiap gambar jika diperlukan investigasi di kemudian hari.
- Mengisi spreadsheet ground truth dengan mereferensikan nama file baru.
- Memulihkan nama file asli jika diperlukan.

Kolom yang tersimpan dalam CSV:

| Kolom | Keterangan |
|---|---|
| `old_path` | Path lengkap file asli |
| `old_filename` | Nama file asli beserta ekstensi |
| `new_filename` | Nama file baru (format `alpr_lampung_XXXXXX.jpg`) |
| `original_extension` | Ekstensi asli file |
| `status` | BERHASIL / GAGAL / DILEWATI |
| `notes` | Catatan tambahan atau pesan error |

In [ ]:
CSV_FIELDNAMES = [
    "old_path",
    "old_filename",
    "new_filename",
    "original_extension",
    "status",
    "notes"
]

with open(CSV_OUTPUT_PATH, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=CSV_FIELDNAMES)
    writer.writeheader()
    writer.writerows(mapping_records)

print(f"[OK] File mapping CSV berhasil disimpan: {CSV_OUTPUT_PATH}")
print(f"     Total record: {len(mapping_records)} baris")

[OK] File mapping CSV berhasil disimpan: /content/drive/MyDrive/renamed_dataset/rename_mapping.csv
     Total record: 185 baris


## Sel 12: Preview Mapping CSV

Beberapa baris pertama dari mapping CSV ditampilkan sebagai verifikasi bahwa data tercatat dengan benar sebelum digunakan lebih lanjut.

In [ ]:
PREVIEW_ROWS = 10

print(f"Preview {min(PREVIEW_ROWS, len(mapping_records))} baris pertama dari mapping CSV:")
print()

# Header
header = f"{'No':>4} | {'Nama File Lama':<40} | {'Nama File Baru':<30} | {'Status':<10} | Catatan"
print(header)
print("-" * len(header))

for i, record in enumerate(mapping_records[:PREVIEW_ROWS]):
    print(
        f"{i+1:>4} | "
        f"{record['old_filename']:<40} | "
        f"{record['new_filename']:<30} | "
        f"{record['status']:<10} | "
        f"{record['notes']}"
    )

if len(mapping_records) > PREVIEW_ROWS:
    print(f"... dan {len(mapping_records) - PREVIEW_ROWS} baris lainnya di file CSV.")

Preview 10 baris pertama dari mapping CSV:

  No | Nama File Lama                           | Nama File Baru                 | Status     | Catatan
-------------------------------------------------------------------------------------------------------
   1 | IMG_20260523_175827.jpg                  | alpr_lampung_000001.jpg        | BERHASIL   | Berhasil
   2 | IMG_20260523_175855.jpg                  | alpr_lampung_000002.jpg        | BERHASIL   | Berhasil
   3 | IMG_20260523_175917.jpg                  | alpr_lampung_000003.jpg        | BERHASIL   | Berhasil
   4 | IMG_20260523_175943.jpg                  | alpr_lampung_000004.jpg        | BERHASIL   | Berhasil
   5 | IMG_20260523_180437.jpg                  | alpr_lampung_000005.jpg        | BERHASIL   | Berhasil
   6 | IMG_20260523_180513.jpg                  | alpr_lampung_000006.jpg        | BERHASIL   | Berhasil
   7 | IMG_20260523_181337.png                  | alpr_lampung_000007.jpg        | BERHASIL   | Berhasil (PNG -> JPG)


## Sel 13: Ringkasan Akhir

Sel terakhir menampilkan ringkasan lengkap hasil proses rename, termasuk lokasi folder output, lokasi file CSV, dan statistik keberhasilan proses.

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("=" * 70)
print("RINGKASAN AKHIR PROSES RENAME DATASET ALPR LAMPUNG")
print("=" * 70)
print()
print(f"Waktu selesai        : {timestamp}")
print()
print("Statistik Proses:")
print(f"  Total file input   : {len(image_files)}")
print(f"  Berhasil diproses  : {success_count}")
print(f"  Gagal diproses     : {failure_count}")
print(f"  Dilewati (skip)    : {skipped_count}")
print()
print("Lokasi Output:")
print(f"  Folder hasil rename: {OUTPUT_FOLDER_PATH}")
print(f"  File mapping CSV   : {CSV_OUTPUT_PATH}")
print()

# Verifikasi jumlah file di folder output
output_jpg_files = list(OUTPUT_FOLDER_PATH.glob(f"{PROJECT_PREFIX}_*.jpg"))
print(f"Verifikasi folder output: {len(output_jpg_files)} file JPG ditemukan.")
print()

if failure_count > 0:
    print("[PERHATIAN] Beberapa file gagal diproses. Periksa kolom 'status' dan 'notes'")
    print("            di file CSV untuk melihat detail error.")
    print()
    failed_records = [r for r in mapping_records if r["status"] == "GAGAL"]
    print("Daftar file yang gagal:")
    for r in failed_records:
        print(f"  - {r['old_filename']}: {r['notes']}")
    print()

print("=" * 70)
print("Proses selesai. Dataset siap digunakan untuk anotasi di Roboflow.")
print("Ground truth (plat, wilayah, kualitas) dapat diisi di spreadsheet terpisah")
print("dengan menggunakan kolom 'new_filename' sebagai kunci referensi.")
print("=" * 70)

RINGKASAN AKHIR PROSES RENAME DATASET ALPR LAMPUNG

Waktu selesai        : 2026-05-23 16:33:03

Statistik Proses:
  Total file input   : 185
  Berhasil diproses  : 185
  Gagal diproses     : 0
  Dilewati (skip)    : 0

Lokasi Output:
  Folder hasil rename: /content/drive/MyDrive/renamed_dataset
  File mapping CSV   : /content/drive/MyDrive/renamed_dataset/rename_mapping.csv

Verifikasi folder output: 185 file JPG ditemukan.

Proses selesai. Dataset siap digunakan untuk anotasi di Roboflow.
Ground truth (plat, wilayah, kualitas) dapat diisi di spreadsheet terpisah
dengan menggunakan kolom 'new_filename' sebagai kunci referensi.
